# Run the full ViralSafeTarget HSV-2 pipeline

This notebook is an orchestration and analysis layer. It calls the existing cached scripts, CLI, and public Python SDK; it does not duplicate alignment, candidate scanning, off-target parsing, ranking, or consensus logic.

The workflow starts with viral genomes, performs quality control and alignment, ranks computational candidates, imports or executes human off-target searches, evaluates pair hypotheses, and compares available tools. Its outputs are prioritization hypotheses. They do **not** prove safety, editing, viral inactivation, treatment efficacy, or a cure.

`SYNTHETIC_MODE=True` runs quickly without genomes or external tools. For the complete cached HSV-2 workflow, change it to `False` and enable the real-data flags below.

## 1. Configuration

In [ ]:
from pathlib import Path
import json
import webbrowser

import matplotlib.pyplot as plt
import pandas as pd
try:
    from IPython.display import FileLink, Markdown, display
except ImportError:  # Lightweight CI execution without the notebook extra
    FileLink = str
    Markdown = str
    def display(value):
        print(value)

import viral_safe_target as vst
from viral_safe_target.notebook_helpers import (
    clear_cache_stamps, detect_cas_offinder, environment_status,
    find_project_root, load_notebook_run, result_funnel,
    run_streaming, safe_read_csv, valid_cas_offinder_output,
)

PROJECT_ROOT = find_project_root()
VIRUS_NAME = 'HSV-2'
SAMPLE_SIZE = 25
CONFIG_YAML = PROJECT_ROOT / 'configs/hsv2_pilot.yaml'
SYNTHETIC_MODE = True
RUN_DOWNLOAD_ALIGNMENT = False
RUN_HUMAN_SCREEN = False
RUN_CONSENSUS = True
OPEN_REPORT = False
FORCE_RERUN = False
CAS_OFFINDER_PATH = None  # Optional explicit executable path
CAS_OFFINDER_DEVICE = 'C'  # CPU/OpenCL device selector used by the completed pilot

print(f'Project: {PROJECT_ROOT}')
print(f'Mode: {"synthetic" if SYNTHETIC_MODE else "real HSV-2"}')

## 2. Environment doctor

A missing optional tool is shown as pending with an actionable command. It is not interpreted as a clean result.

In [ ]:
doctor = environment_status(PROJECT_ROOT)
display(doctor)
if doctor['status'].isin(['missing', 'warning', 'pending']).any():
    display(Markdown('**Review pending/warning rows before a real-data run.**'))

## 3. Full cached real-data workflow

In real mode this calls `scripts/run_real_hsv2.sh`. Output is streamed and failures stop the notebook. Expensive stages remain checksum-cached. `FORCE_RERUN` removes only JSON cache stamps inside approved run `.cache` directories.

In [ ]:
if FORCE_RERUN and not SYNTHETIC_MODE:
    removed = clear_cache_stamps(PROJECT_ROOT, [
        'reports/real_hsv2/.cache',
        'reports/hsv2_pilot/.cache',
        'reports/hsv2_consensus/.cache',
    ])
    print(f'Removed {len(removed)} cache stamps; source data and outputs were preserved.')

if SYNTHETIC_MODE:
    run_streaming(['bash', 'scripts/run_demo.sh'], cwd=PROJECT_ROOT)
elif RUN_DOWNLOAD_ALIGNMENT:
    command = [
        'bash', 'scripts/run_real_hsv2.sh',
        '--sample-size', str(SAMPLE_SIZE),
        '--config', str(CONFIG_YAML),
    ]
    if RUN_HUMAN_SCREEN:
        command.insert(2, '--with-human')
    run_streaming(command, cwd=PROJECT_ROOT)
else:
    print('Real-data download/alignment stage skipped; cached files will be used if present.')

## 4. Focused HSV-2 pilot

The pilot selects UL19/UL30 candidates and builds or summarizes the human-screen exchange files. Synthetic mode uses the bundled demonstration workflow.

In [ ]:
if not SYNTHETIC_MODE:
    run_streaming(['bash', 'scripts/run_hsv2_pilot.sh'], cwd=PROJECT_ROOT)

pilot_data = load_notebook_run(PROJECT_ROOT, synthetic=SYNTHETIC_MODE)
qc_counts = pilot_data['qc']['decision'].value_counts().rename_axis('decision').reset_index(name='count')
selected_counts = pilot_data['selected']['gene_name'].value_counts().rename_axis('gene').reset_index(name='selected')
summary = pd.DataFrame([
    {'metric': 'Accepted genomes', 'count': int((pilot_data['qc']['decision'] == 'accepted').sum())},
    {'metric': 'Rejected genomes', 'count': int((pilot_data['qc']['decision'] == 'rejected').sum())},
    {'metric': 'Initial candidates', 'count': len(pilot_data['pre_human']) + len(pilot_data['rejected_pre_human'])},
    {'metric': 'Retained pre-human', 'count': len(pilot_data['pre_human'])},
    {'metric': 'Rejected pre-human', 'count': len(pilot_data['rejected_pre_human'])},
])
display(summary)
display(selected_counts)

## 5. Human off-target execution

Cas-OFFinder is resolved in this order: the configured path, `tools/bin/cas-offinder`, then `PATH`. It runs only when the existing output is absent/invalid or `FORCE_RERUN=True`. Missing execution remains pending.

In [ ]:
cas_status = 'not requested'
if not SYNTHETIC_MODE and RUN_HUMAN_SCREEN:
    cas_input = PROJECT_ROOT / 'reports/hsv2_pilot/cas_offinder_input.txt'
    cas_output = PROJECT_ROOT / 'reports/hsv2_pilot/cas_offinder_output.tsv'
    cas_executable = detect_cas_offinder(PROJECT_ROOT, CAS_OFFINDER_PATH)
    must_run = FORCE_RERUN or not valid_cas_offinder_output(cas_output)
    if must_run and not cas_input.is_file():
        raise FileNotFoundError(f'Cas-OFFinder input is missing: {cas_input}. Run the pilot stage first.')
    if must_run and cas_executable is None:
        cas_status = 'pending: executable unavailable; missing output is not a zero-hit result'
    elif must_run:
        run_streaming([cas_executable, cas_input, CAS_OFFINDER_DEVICE, cas_output], cwd=PROJECT_ROOT)
        cas_status = 'completed'
    else:
        cas_status = 'cached valid output'
    if valid_cas_offinder_output(cas_output):
        run_streaming(['bash', 'scripts/run_hsv2_pilot.sh'], cwd=PROJECT_ROOT)
elif SYNTHETIC_MODE:
    cas_status = 'synthetic small-host screen; genome-scale Cas-OFFinder not required'
else:
    cas_status = 'pending by configuration'
print(cas_status)

## 6. Result loading through the public SDK

In [ ]:
if SYNTHETIC_MODE:
    data = load_notebook_run(PROJECT_ROOT, synthetic=True)
    run = data['run']
else:
    run = vst.load_run(PROJECT_ROOT / 'reports/hsv2_pilot')
    data = load_notebook_run(PROJECT_ROOT, synthetic=False)

pre_human_candidates = data['pre_human']
post_human_candidates = run.candidates
predicted_human_hits = run.human_hits
same_gene_pairs = data['same_gene_pairs']
multi_target_pairs = data['multi_target_pairs']
manifest = run.manifest

display(pd.DataFrame([{
    'run_directory': str(data['run_directory']),
    'pre_human_candidates': len(pre_human_candidates),
    'post_human_candidates': len(post_human_candidates),
    'predicted_human_hits': len(predicted_human_hits),
    'same_gene_pairs': len(same_gene_pairs),
    'multi_target_pairs': len(multi_target_pairs),
    'manifest_git_commit': manifest.get('git_commit', 'not recorded'),
}]))

## 7. Result funnel

In [ ]:
funnel = result_funnel(data)
display(funnel)
ax = funnel.sort_values('count').plot.barh(x='stage', y='count', legend=False, figsize=(9, 4))
ax.set_title(f'{VIRUS_NAME} computational result funnel')
ax.set_xlabel('Records or candidates')
plt.tight_layout()
if plt.get_backend().lower() == 'agg':
    plt.close()
else:
    plt.show()

## 8. Candidate interpretation

A candidate with no predicted hit within the configured model and mismatch threshold is **not** thereby safe or experimentally effective. It remains a computational candidate requiring expert review and experimental validation.

In [ ]:
candidate_view = post_human_candidates.copy().reset_index(drop=True)
candidate_view.insert(0, 'rank', range(1, len(candidate_view) + 1))
candidate_columns = {
    'rank': 'Rank', 'candidate_id': 'Candidate ID', 'gene_name': 'Gene',
    'exact_strain_coverage': 'Guide coverage across strains',
    'gc_fraction': 'GC fraction',
    'reference_viral_occurrence_count': 'Reference viral occurrences',
    'human_exact_hit_count': 'Exact human hits',
    'human_one_mismatch_hit_count': '1-mismatch human hits',
    'human_two_mismatch_hit_count': '2-mismatch human hits',
    'human_three_mismatch_hit_count': '3-mismatch human hits',
    'pre_human_score': 'Pre-human score', 'post_human_score': 'Post-human score',
    'decision': 'Decision', 'decision_reason': 'Explanation',
}
for source in candidate_columns:
    if source not in candidate_view:
        candidate_view[source] = pd.NA
display(candidate_view[list(candidate_columns)].rename(columns=candidate_columns).head(20))

## 9. UL19 versus UL30 descriptive analysis

These are descriptive counts and percentages. They do not establish a causal difference between genes. Synthetic mode displays its demonstration genes instead.

In [ ]:
analysis_candidates = post_human_candidates.copy()
analysis_candidates['human_total_predicted_hits'] = pd.to_numeric(
    analysis_candidates.get('human_total_predicted_hits'), errors='coerce'
).fillna(0)
selected_by_gene = data['selected']['gene_name'].value_counts()
gene_summary = analysis_candidates.groupby('gene_name', dropna=False).agg(
    evaluated_candidates=('candidate_id', 'count'),
    retained_candidates=('decision', lambda values: values.astype(str).eq('retain_computational_candidate').sum()),
    total_predicted_human_hits=('human_total_predicted_hits', 'sum'),
    no_predicted_hit_count=('human_total_predicted_hits', lambda values: values.eq(0).sum()),
).reset_index()
gene_summary.insert(1, 'selected_candidates', gene_summary['gene_name'].map(selected_by_gene).fillna(0).astype(int))
gene_summary['no_predicted_hit_percent'] = 100 * gene_summary['no_predicted_hit_count'] / gene_summary['evaluated_candidates']
display(gene_summary)

## 10. Pair hypotheses

Same-gene rows describe idealized sequence-interval hypotheses. A cross-gene pair is an independent multi-target hypothesis, **not one physical deletion**. Neither table predicts editing, repair, delivery, toxicity, viral inactivation, or clinical outcome.

In [ ]:
pair_columns = [
    'candidate_a', 'candidate_b', 'gene_a', 'gene_b', 'distance_bp',
    'deletion_length_bp', 'joint_strain_coverage', 'pair_score', 'limitations',
]
def pair_view(frame):
    if frame.empty:
        return pd.DataFrame(columns=pair_columns)
    result = frame.copy()
    for column in pair_columns:
        if column not in result:
            result[column] = pd.NA
    return result.sort_values('pair_score', ascending=False, na_position='last')[pair_columns].head(20)

display(Markdown('### Top 20 same-gene hypotheses'))
display(pair_view(same_gene_pairs))
display(Markdown('### Top 20 multi-target hypotheses'))
display(pair_view(multi_target_pairs))

## 11. Multi-tool comparison

Raw scores from unrelated tools are never averaged. The consensus layer compares within-tool ranks and percentiles, exposes coverage and disagreement, and preserves pending tools.

In [ ]:
consensus_dir = PROJECT_ROOT / 'reports/hsv2_consensus'
if not SYNTHETIC_MODE and RUN_CONSENSUS:
    run_streaming(['bash', 'scripts/run_hsv2_consensus.sh'], cwd=PROJECT_ROOT)

if not SYNTHETIC_MODE and (consensus_dir / 'consensus_candidates.csv').is_file():
    tool_coverage = safe_read_csv(consensus_dir / 'tool_coverage.csv', ['tool_name', 'status'], label='Tool coverage')
    consensus_ranking = safe_read_csv(consensus_dir / 'consensus_candidates.csv', ['candidate_id', 'consensus_rank'], label='Consensus ranking')
    model_agreement = safe_read_csv(consensus_dir / 'model_agreement.csv', label='Model agreement')
    disagreement = safe_read_csv(consensus_dir / 'disagreement_report.csv', label='Disagreement report')
else:
    baseline = vst.candidate_metrics_as_tool_results(post_human_candidates)
    comparison = vst.compare_tools(
        post_human_candidates, [baseline],
        expected_tools=[
            'viral_safe_target_pre_human', 'viral_safe_target_post_human',
            'cas-offinder', 'crispritz', 'crispor', 'chopchop', 'guidescan2',
        ],
    )
    tool_coverage = comparison.tool_coverage
    consensus_ranking = comparison.consensus_candidates
    model_agreement = comparison.model_agreement
    disagreement = comparison.disagreement_report

display(Markdown('### Tool coverage and pending stages'))
display(tool_coverage)
display(Markdown('### Consensus ranking'))
display(consensus_ranking.head(20))
display(Markdown('### Rank correlations and top-k overlap'))
display(model_agreement)
display(Markdown('### High-disagreement or missing-tool candidates'))
display(disagreement.head(20))

## 12. Report and machine-readable output links

In [ ]:
run_directory = Path(data['run_directory'])
link_names = [
    'report.html', 'candidates_ranked_post_human.csv', 'candidates.csv',
    'predicted_human_hits.csv', 'pair_hypotheses_same_gene.csv',
    'pair_hypotheses_multi_target.csv', 'simulated_pairs.csv',
    'run_manifest.json', 'methods.md', 'limitations.md',
]
available_links = [run_directory / name for name in link_names if (run_directory / name).is_file()]
for path in available_links:
    display(FileLink(str(path)))
if not SYNTHETIC_MODE and (consensus_dir / 'report.html').is_file():
    display(FileLink(str(consensus_dir / 'report.html')))

primary_report = run_directory / 'report.html'
if OPEN_REPORT and primary_report.is_file():
    webbrowser.open(primary_report.resolve().as_uri())

## 13. Research interpretation

### What was learned
The run provides a reproducible, provenance-linked prioritization of conserved viral candidate sites, their modeled human-hit burden, pair hypotheses, and the agreement or disagreement among available computational models.

### What remains unknown
The workflow does not establish cellular accessibility, editing frequency, repair outcomes, delivery, toxicity, viral viability, latency effects, safety, or clinical efficacy. A missing external-tool result remains missing.

### Next computational experiment
Complete the pending CRISPRitz search, import documented CRISPOR/CHOPCHOP/GuideScan2 exports, and examine candidates whose rank changes or whose variant/bulge burden disagrees with Cas-OFFinder.

### External validation
Candidate interpretation requires domain-expert review. Any claim about measured editing or biological effect requires appropriate independently designed experiments and sequencing analysis; future CRISPResso2 measurements must remain separate from prediction scores.